PDF 파일기반 질의응답 챗봇 (랭체인, 그라디오, Upstage)

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:4])

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])

gsk_
JF


In [ ]:
# %pip install -q pypdf
# %pip install -q faiss-cpu
# %pip install -q tiktoken

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


c:\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# PDF 로드
loader = PyPDFLoader("../data/tutorial-korean.pdf")
documents = loader.load()
print(len(documents))

39


In [4]:
# 텍스트 분할
#text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=0,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)
#texts = text_splitter.split_documents(documents)
texts = loader.load_and_split(text_splitter=text_splitter)
print(len(texts))

284


In [5]:
from langchain_upstage import UpstageEmbeddings

# OpenAI Embeddings 적용
#embeddings = OpenAIEmbeddings()
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

# FAISS 벡터 저장소 생성
vector_store = FAISS.from_documents(texts, embeddings)
vector_store.save_local("../db/faiss_db")
# 벡터 저장소에서 검색하기
# search_type="similarity" 옵션은 retrieval 객체에서 유사성 검색을 사용하여 질문 vector와 가장 유사한 문장 vector를 선택함
# search_kwargs 옵션은 vector 저장소에서 Prompt 2개의 텍스트 덩어리로 보내려는 것을 의미함
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 6}
)

In [6]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# ChatOpenAI는 기본모델인  gpt-3.5-turbo 사용하고
# temperature=0은 보수적인 지문에 대한 답변을 내고, temperature=1 다양한 답변을 낼 수 있음 
#llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
)
print(llm.model_name)

solar-pro


In [8]:
from langchain_core.prompts import PromptTemplate

# 한국어 최적화 프롬프트
prompt_template = """
당신은 BlueJ 프로그래밍 환경 전문가입니다. 
아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

문서 내용:
{context}

질문: {question}

답변 규칙:
1. 문서 내용만을 근거로 답변하세요
2. 단계별 설명이 필요하면 순서대로 작성하세요  
3. 구체적인 메뉴명, 버튼명을 포함하세요
4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요

답변:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)
print(" 프롬프트 설정 완료")
print(prompt)

 프롬프트 설정 완료
input_variables=['context', 'question'] input_types={} partial_variables={} template='\n당신은 BlueJ 프로그래밍 환경 전문가입니다. \n아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.\n\n문서 내용:\n{context}\n\n질문: {question}\n\n답변 규칙:\n1. 문서 내용만을 근거로 답변하세요\n2. 단계별 설명이 필요하면 순서대로 작성하세요  \n3. 구체적인 메뉴명, 버튼명을 포함하세요\n4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요\n\n답변:'


In [ ]:

# RetrievalQA가 실질적인 RAG를 수행하는 객체
# chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=retriever,  # 기존 retriever 유지
#     chain_type_kwargs={"prompt": prompt},
#     return_source_documents=True
# )
# chain

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n당신은 BlueJ 프로그래밍 환경 전문가입니다. \n아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.\n\n문서 내용:\n{context}\n\n질문: {question}\n\n답변 규칙:\n1. 문서 내용만을 근거로 답변하세요\n2. 단계별 설명이 필요하면 순서대로 작성하세요  \n3. 구체적인 메뉴명, 버튼명을 포함하세요\n4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요\n\n답변:'), llm=ChatUpstage(client=<openai.resources.chat.completions.completions.Completions object at 0x000001A2708EA750>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001A2708EADB0>, model_name='solar-pro', temperature=0.5, model_kwargs={}, upstage_api_key=SecretStr('**********'), upstage_api_base='https://api.upstage.ai/v1'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template=

In [9]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LCEL RAG chain using RunnablePassthrough
def format_docs(docs):
    """Format retrieved docs as context string"""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL RAG chain 설정 완료")

# Invoke with question (returns answer string)
query = "코드패드는 무엇이고 어떻게 사용하나요?"
query_result = rag_chain.invoke(query)
print(query_result)

# To get source documents, access retriever separately or modify chain
source_docs = retriever.invoke(query)
print("Source documents:", [doc.metadata for doc in source_docs])

LCEL RAG chain 설정 완료
**코드패드란?**  
코드패드는 BlueJ에서 자바 코드(표현식 및 명령문)의 일부를 빠르게 평가하고 테스트할 수 있는 도구입니다. 코드의 의미를 분석하거나 구문을 시험하는 데 유용합니다.  

**사용 방법 (단계별):**  
1. **코드패드 열기**  
   - BlueJ 메뉴 바에서 **보기(Show) > 코드패드보기(Show Code Pad)**를 선택합니다.  

2. **코드 입력 및 실행**  
   - 코드패드 영역에 자바 표현식 또는 명령문을 입력합니다.  
     *예시: `2 + 3`, `System.out.println("Hello")`*  
   - **Enter 키**를 눌러 코드를 실행합니다. 결과는 즉시 화면에 표시됩니다.  

3. **입력 이력 활용**  
   - 이전 입력 내용을 재사용하려면 **상화 화살표 키(↑/↓)**를 사용합니다.  
   - 이력에서 선택한 내용을 수정 후 재실행할 수 있습니다.  

**주의 사항**  
- 코드패드는 전체 프로그램이 아닌 **코드 조각**만 평가할 수 있습니다.  
- 문서에 명시되지 않은 기능(예: 디버깅 지원)은 "문서에서 찾을 수 없습니다".  

> 그림 13을 참조하면 코드패드의 UI 위치를 확인할 수 있습니다.
Source documents: [{'producer': 'Acrobat Distiller with ezUniHFT', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2005-04-26T15:21:34+09:00', 'moddate': '2005-04-26T15:21:34+09:00', 'author': 'Owner', 'title': '<426C75654AC7D1B1DBC6A9C5E4B8AEBEF3B9AEBCAD283230292E687770>', 'source': '../data/tutorial-korean.pdf', 'total_pages': 39, 'page': 

In [10]:
query = "애플릿은 무엇이고 어떻게 사용하나요?"
query_result = rag_chain.invoke(query)
print(type(query_result))
print(query_result)

<class 'str'>
**애플릿(Applet)이란?**  
문서에서 명시적으로 정의되진 않았으나, BlueJ 환경에서 브라우저 기반으로 실행되는 작은 Java 프로그램을 의미합니다. (문서 32~33페이지 참조)

---

### **애플릿 사용 방법 (단계별 설명)**  
1. **애플릿 만들기**  
   - 문서 9.2절(33페이지)에 따라, BlueJ에서 `Applet` 클래스를 상속받는 클래스를 생성합니다.  
     - 예: `public class MyApplet extends java.applet.Applet`  
   - 애플릿의 생명주기 메서드(`init()`, `paint(Graphics g)` 등)를 오버라이드하여 기능을 구현합니다.

2. **HTML 파일 생성**  
   - 문서에 구체적 설명은 없으나, 애플릿을 실행하려면 HTML 파일에 `<applet>` 태그로 호출해야 합니다.  
     - 예: `<applet code="MyApplet.class" width="200" height="200"></applet>`

3. **애플릿 실행하기**  
   - **9.1절(32페이지) 참조**: BlueJ에서 애플릿 클래스를 우클릭 → **"애플릿 실행"** 메뉴 선택.  
   - Windows/macOS: 기본 브라우저가 자동으로 열립니다.  
   - Unix: BlueJ 설정 파일에 정의된 브라우저를 사용합니다.

4. **애플릿 테스트하기**  
   - **9.3절(33페이지) 참조**: 브라우저 창에서 애플릿의 동작을 확인합니다.  
   - 오류 발생 시, BlueJ의 컴파일 오류 메시지나 브라우저 콘솔 로그를 확인합니다.

---

### **주의 사항**  
- 문서에는 애플릿의 보안 제한, 최신 브라우저 호환성 문제 등은 언급되지 않았습니다. ("문서에서 찾을 수 없습니다")  
- HTML 작성법은 직접 구현해야 하며, 문서 11.6절(38페이지)에서 추가 예제를 참고할 수 있습니다.  

> 💡 **문

In [ ]:
#%pip install -q gradio

In [34]:
from langchain.document_loaders import PyPDFLoader
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import ChatPromptTemplate
import gradio as gr

# 1. PDF를 한 번만 로드하여 벡터 저장소 생성
def initialize_retriever():
    pdf_path = "../data/tutorial-korean.pdf"
    # PDF 파일을 로드하여 문서 객체로 변환
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # 문서를 일정한 크기로 분할 (chunk_size=1000, 중첩 없음)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200, 
        chunk_overlap=0,
        separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
    )
    texts = loader.load_and_split(text_splitter=text_splitter)

    # 문서의 텍스트를 벡터 임베딩으로 변환
    embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

    # 변환된 벡터를 FAISS 벡터 저장소에 저장
    vector_store = FAISS.from_documents(texts, embeddings)
    vector_store.save_local("../db/faiss_db")
    # 저장된 벡터를 검색할 수 있는 retriever 생성 (유사한 문서 2개 검색)
    retriever = vector_store.as_retriever(
        search_type="similarity", 
        search_kwargs={"k": 6}
    )
    return retriever

# 2. 전역 retriever 생성 (앱 시작 시 한 번만 실행)
retriever = initialize_retriever()

retrieved_docs = retriever.invoke("코드패드는 무엇이고 어떻게 사용하나요?")
for doc in retrieved_docs:
    print(doc.page_content)

6. 코드패드 사용하기 ··················································· ······························································· ········· 21
216. 코드패드의 사용
BlueJ 코드패드는 자바 코드(표현식과 명령문)의 일부분을 쉽고 빠르게 평가할 수 있는 기능을 
제공합니다. 따라서, 코드패드는 자바언어로 작성된 프로그램 코드의 의미를 상세히 조사하거나 구문을 예증하고 시험하는데 사용할 수 있 습니다.
6.1. 코드패드 나타내기
요 약 코드패드에 명령문들을 입력하여 실행시킬 수 있습니다.
코드패드에서는 사용자가 이전에 입력한 내용들에 대한 이력을 저장하고 있습니다. 상하 화살표를 
이용하여 이전에 입력한 내용을 손쉽게 다시 입력시킬 수 있을 뿐만 아니라, 이전에 입력했던 내용을 수정하여 재입력시킬 수도 있습니다.   요 약 입력이력을 사용하기 위해서는 상하 화살표를 사용하십시오.
6.1. 코드패드 보기 ················································ ······························································· ··········· 21
코드패드 영역에는 표현식 또는 문장들을 입력할 수 있습니다. 키보드의 Enter를 누르면 
입력된 표현식 또는 문장이 라인단위로 평가되어 그 결과가 화면에 나타납니다.
그림 13 : 코드패드를 나타낸 메인화면  요 약  코드패드를 사용하기 위해 보기메뉴에서 코드패드보기(Show Code Pad)를 
 선택하십시오


In [ ]:

# 3. 채팅 응답 함수 (retriever를 재사용)
def chat_respond(message, chat_history):
    #  시스템 메시지 템플릿: 문서 요약을 기반으로 질문에 답변하도록 설정
    system_template = """당신은 BlueJ 프로그래밍 환경 전문가입니다. 
        아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

        문서 내용:
        {context}

        질문: {question}

        답변 규칙:
        1. 문서 내용만을 근거로 답변하세요
        2. 단계별 설명이 필요하면 순서대로 작성하세요  
        3. 구체적인 메뉴명, 버튼명을 포함하세요
        4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요

        답변:
    """
    # 사용자 질문을 받아 최종 Prompt 구성
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_template),
        ("human", "{question}")
    ])

    chain_type_kwargs = {
        "prompt": prompt, #  LLM이 사용할 프롬프트 지정
        "document_variable_name": "context", #  문서 요약 데이터를 LLM에 전달할 변수 이름
    }
    
    llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
    
    # RetrievalQA 체인 생성
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",  #  문서를 하나의 큰 텍스트로 처리하는 방식
        retriever=retriever, #  유사 문서를 검색하는 retriever 연결
        return_source_documents=True, #  답변에 참조한 문서 정보 포함
        chain_type_kwargs=chain_type_kwargs,
        input_key="question" #  사용자 질문을 "question" 키로 전달
    )
    
    # LLM을 호출하여 질문에 대한 응답 생성
    query_result = chain.invoke({"question": message})

    # 모델의 답변을 가져오기
    bot_message = query_result['result']

    # 참조한 문서 정보를 응답에 추가
    for i, doc in enumerate(query_result['source_documents']):
        bot_message += f' [{i+1}] {doc.metadata.get("source", "Unknown")} (Page {doc.metadata.get("page", "N/A")})'

    # Gradio 채팅 기록 형식에 맞춰 응답을 저장
    chat_history.append({"role": "user", "content": message})  # 사용자 메시지 추가
    chat_history.append({"role": "assistant", "content": bot_message})  # 봇 응답 추가
    
    # 입력 창 초기화 및 갱신된 채팅 기록 반환
    return "", chat_history

# 4. # Gradio UI 생성 및 실행
with gr.Blocks() as demo:  
    # 채팅 창 (채팅 메시지를 표시하는 Gradio 컴포넌트)
    chatbot = gr.Chatbot(label="채팅창", type="messages")
    # 사용자 입력 텍스트 박스 (메시지를 입력하는 필드)
    msg = gr.Textbox(label="입력")
    # 초기화 버튼 (채팅 기록을 초기화하는 버튼)
    clear = gr.Button("초기화")

    # 사용자가 입력 필드에 메시지를 입력하면 chat_respond()가 실행됨
    #  - 입력한 메시지는 msg에서 가져옴
    #  - 기존 채팅 기록(chatbot)과 함께 전달됨
    #  - 응답을 받은 후, msg를 초기화하고 chatbot에 대화 내용을 추가
    msg.submit(chat_respond, [msg, chatbot], [msg, chatbot])  
    # 초기화 버튼 클릭 시, 채팅 기록을 비우도록 설정
    clear.click(lambda: [], None, chatbot, queue=False)

# Gradio 앱 실행 (debug 모드 활성화)
demo.launch(debug=True)

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import gradio as gr
from operator import itemgetter

# 1. PDF를 한 번만 로드하여 벡터 저장소 생성 (변경 없음)
def initialize_retriever():
    pdf_path = "../data/tutorial-korean.pdf"
    # PDF 파일을 로드하여 문서 객체로 변환
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # 문서를 일정한 크기로 분할 (chunk_size=1000, 중첩 없음)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200, 
        chunk_overlap=0,
        separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
    )
    texts = loader.load_and_split(text_splitter=text_splitter)

    # 문서의 텍스트를 벡터 임베딩으로 변환
    embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

    # 변환된 벡터를 FAISS 벡터 저장소에 저장
    vector_store = FAISS.from_documents(texts, embeddings)
    vector_store.save_local("../db/faiss_db")
    # 저장된 벡터를 검색할 수 있는 retriever 생성 (유사한 문서 6개 검색)
    retriever = vector_store.as_retriever(
        search_type="similarity", 
        search_kwargs={"k": 6}
    )
    return retriever

# 2. 전역 retriever 생성 (앱 시작 시 한 번만 실행)
retriever = initialize_retriever()

retrieved_docs = retriever.invoke("코드패드는 무엇이고 어떻게 사용하나요?")
for doc in retrieved_docs:
    print(doc.page_content)


6. 코드패드 사용하기 ··················································· ······························································· ········· 21
216. 코드패드의 사용
BlueJ 코드패드는 자바 코드(표현식과 명령문)의 일부분을 쉽고 빠르게 평가할 수 있는 기능을 
제공합니다. 따라서, 코드패드는 자바언어로 작성된 프로그램 코드의 의미를 상세히 조사하거나 구문을 예증하고 시험하는데 사용할 수 있 습니다.
6.1. 코드패드 나타내기
요 약 코드패드에 명령문들을 입력하여 실행시킬 수 있습니다.
코드패드에서는 사용자가 이전에 입력한 내용들에 대한 이력을 저장하고 있습니다. 상하 화살표를 
이용하여 이전에 입력한 내용을 손쉽게 다시 입력시킬 수 있을 뿐만 아니라, 이전에 입력했던 내용을 수정하여 재입력시킬 수도 있습니다.   요 약 입력이력을 사용하기 위해서는 상하 화살표를 사용하십시오.
6.1. 코드패드 보기 ················································ ······························································· ··········· 21
코드패드 영역에는 표현식 또는 문장들을 입력할 수 있습니다. 키보드의 Enter를 누르면 
입력된 표현식 또는 문장이 라인단위로 평가되어 그 결과가 화면에 나타납니다.
그림 13 : 코드패드를 나타낸 메인화면  요 약  코드패드를 사용하기 위해 보기메뉴에서 코드패드보기(Show Code Pad)를 
 선택하십시오


In [ ]:

# 3. 채팅 응답 함수 (LCEL + RunnablePassthrough로 변경)
def chat_respond(message, chat_history):
    # 시스템 메시지 템플릿: 문서 요약을 기반으로 질문에 답변하도록 설정
    system_template = """당신은 BlueJ 프로그래밍 환경 전문가입니다. 
        아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

        문서 내용:
        {context}

        질문: {question}

        답변 규칙:
        1. 문서 내용만을 근거로 답변하세요
        2. 단계별 설명이 필요하면 순서대로 작성하세요  
        3. 구체적인 메뉴명, 버튼명을 포함하세요
        4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요

        답변:
    """
    # 사용자 질문을 받아 최종 Prompt 구성
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_template),
        ("human", "{question}")
    ])

    llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
    
    # LCEL 체인 구성: retriever → 문서 포맷팅 → prompt → llm → 출력 파싱
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    # LLM을 호출하여 질문에 대한 응답 생성
    bot_message = chain.invoke(message)

    # 참조 문서 정보를 별도로 가져와 응답에 추가
    source_docs = retriever.invoke(message)
    for i, doc in enumerate(source_docs):
        bot_message += f' [{i+1}] {doc.metadata.get("source", "Unknown")} (Page {doc.metadata.get("page", "N/A")})'

    # Gradio 채팅 기록 형식에 맞춰 응답을 저장
    chat_history.append({"role": "user", "content": message})  # 사용자 메시지 추가
    chat_history.append({"role": "assistant", "content": bot_message})  # 봇 응답 추가
    
    # 입력 창 초기화 및 갱신된 채팅 기록 반환
    return "", chat_history

# 4. Gradio UI 생성 및 실행 (변경 없음)
with gr.Blocks() as demo:  
    # 채팅 창 (채팅 메시지를 표시하는 Gradio 컴포넌트)
    chatbot = gr.Chatbot(label="채팅창")
    # 사용자 입력 텍스트 박스 (메시지를 입력하는 필드)
    msg = gr.Textbox(label="입력")
    # 초기화 버튼 (채팅 기록을 초기화하는 버튼)
    clear = gr.Button("초기화")

    # 사용자가 입력 필드에 메시지를 입력하면 chat_respond()가 실행됨
    msg.submit(chat_respond, [msg, chatbot], [msg, chatbot])  
    # 초기화 버튼 클릭 시, 채팅 기록을 비우도록 설정
    clear.click(lambda: [], None, chatbot, queue=False)

# Gradio 앱 실행 (debug 모드 활성화)
demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
